# 02_dim_producto

**Origen:** `02_dim_producto.ktr` (transformación de Pentaho)  
**Fecha:** 2026-10-27  
**Biblioteca:** pandas  
**Cobertura:** Steps: 5 (5 exactos, 0 aproximados, 0 a completar)

*Generado por Pentaho Live a partir de la transformación.*

## Cómo correrlo

1. Instala las bibliotecas: `pip install pandas numpy openpyxl matplotlib`
2. Deja este archivo en la carpeta del .ktr (las rutas relativas parten de ahí).
3. Variables de entorno: `PDI_PASSWORD_DW_GRANO_DE_ORO` (las contraseñas nunca van en el código).

In [ ]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# carpeta del script (o del notebook): la misma que la del .ktr
AQUI = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
VARS = {
    "Internal.Transformation.Filename.Directory": os.environ.get("Internal.Transformation.Filename.Directory", AQUI),
    "Internal.Entry.Current.Directory": os.environ.get("Internal.Entry.Current.Directory", AQUI),
    "Internal.Transformation.Name": "02_dim_producto",
}

engine_dw_grano_de_oro = create_engine("sqlite:///" + f"{VARS['Internal.Entry.Current.Directory']}/../salida/grano_de_oro_dw.sqlite")

import matplotlib.pyplot as plt

## 1. Leer productos (Excel)

`ExcelInput`

Lee productos.xlsx (4 campos)

> Nota: Necesita openpyxl (xlsx) o xlrd (xls antiguos).

In [ ]:
archivos_leer_productos_excel = [f"{VARS['Internal.Entry.Current.Directory']}/../datos/productos.xlsx"]
partes = []
for archivo in archivos_leer_productos_excel:
    for hoja, fila0, col0 in [("Productos", 0, 0)]:
        parte = pd.read_excel(archivo, sheet_name=hoja, header=None, skiprows=fila0 + 1, dtype=object)
        parte = parte.iloc[:, col0:col0 + 4]
        parte.columns = ["codigo_producto", "nombre", "categoria", "precio_venta"][:parte.shape[1]]
        partes.append(parte)
df_leer_productos_excel = pd.concat(partes, ignore_index=True)
df_leer_productos_excel = df_leer_productos_excel.dropna(how="all", subset=["codigo_producto", "nombre", "categoria", "precio_venta"]).reset_index(drop=True)
df_leer_productos_excel["codigo_producto"] = df_leer_productos_excel["codigo_producto"].astype("string")
df_leer_productos_excel["nombre"] = df_leer_productos_excel["nombre"].astype("string")
df_leer_productos_excel["categoria"] = df_leer_productos_excel["categoria"].astype("string")
df_leer_productos_excel["precio_venta"] = pd.to_numeric(df_leer_productos_excel["precio_venta"]).astype("float64")
df_leer_productos_excel.head(10)

## 2. Estandarizar textos

`StringOperations`

Limpia campos de texto

In [ ]:
df_estandarizar_textos = df_leer_productos_excel.copy()
df_estandarizar_textos["codigo_producto"] = df_estandarizar_textos["codigo_producto"].str.strip(" ").str.upper()
df_estandarizar_textos["nombre"] = df_estandarizar_textos["nombre"].str.strip(" ")
df_estandarizar_textos["categoria"] = df_estandarizar_textos["categoria"].str.strip(" ").str.title()
df_estandarizar_textos.head(10)

## 3. Ordenar por código

`SortRows`

Ordena por codigo_producto

In [ ]:
df_ordenar_por_codigo = df_estandarizar_textos.sort_values(["codigo_producto"], ascending=True, na_position="first", kind="stable", key=lambda c: c.str.lower() if c.name in ["codigo_producto"] else c)
df_ordenar_por_codigo = df_ordenar_por_codigo.reset_index(drop=True)
df_ordenar_por_codigo.head(10)

## 4. Asignar clave subrogada

`Sequence`

Step de tipo Sequence

In [ ]:
df_asignar_clave_subrogada = df_ordenar_por_codigo.copy()
df_asignar_clave_subrogada["sk_producto"] = pd.Series(1 + np.arange(len(df_asignar_clave_subrogada)) * 1, index=df_asignar_clave_subrogada.index, dtype="Int64")
df_asignar_clave_subrogada.head(10)

## 5. Cargar dim_producto

`TableOutput`

Escribe en la tabla dim_producto (dw_grano_de_oro)

In [ ]:
df_cargar_dim_producto = df_asignar_clave_subrogada
with engine_dw_grano_de_oro.begin() as _con:
    _con.execute(text("DELETE FROM dim_producto"))  # vaciar la tabla (truncate)
df_asignar_clave_subrogada[["sk_producto", "codigo_producto", "nombre", "categoria", "precio_venta"]].to_sql("dim_producto", engine_dw_grano_de_oro, if_exists="append", index=False, chunksize=1000)
df_cargar_dim_producto.head(10)

## Análisis exploratorio

Para cada entrada y cada salida: forma, tipos, nulos, estadísticas, histogramas de las columnas numéricas y las categorías más frecuentes.

In [ ]:
def eda(df, titulo, max_filas=10000):
    """Análisis exploratorio de un DataFrame: forma, tipos, nulos, estadísticas y gráficos."""
    print(f"== {titulo}: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(pd.DataFrame({"dtype": df.dtypes.astype(str), "nulos": df.isna().sum()}))
    if df.empty:
        return
    print(df.describe(include="all").T)
    numericas = df.select_dtypes(include="number").columns.tolist()
    if numericas:
        df[numericas].hist(bins=20, figsize=(4 * min(len(numericas), 3), 3 * ((len(numericas) + 2) // 3)), layout=((len(numericas) + 2) // 3, min(len(numericas), 3)))
        plt.suptitle(titulo + " - distribución de las columnas numéricas")
        plt.tight_layout()
        plt.show()
    texto = [c for c in df.columns if c not in numericas][:3]
    for c in texto:
        top = df[c].astype(str).value_counts().head(10)
        if 1 < len(top) and top.iloc[0] > 1:
            top.sort_values().plot.barh(figsize=(6, 3), title=f"{titulo} - {c} (top 10)")
            plt.tight_layout()
            plt.show()

### Entrada «Leer productos (Excel)»

In [ ]:
eda(df_leer_productos_excel, "Entrada «Leer productos (Excel)»")

### Salida «Cargar dim_producto»

In [ ]:
eda(df_cargar_dim_producto, "Salida «Cargar dim_producto»")

## Resumen

Steps: 5 (5 exactos, 0 aproximados, 0 a completar)

| # | Step | Tipo | Estado |
|---|---|---|---|
| 1 | Leer productos (Excel) | `ExcelInput` | exacto |
| 2 | Estandarizar textos | `StringOperations` | exacto |
| 3 | Ordenar por código | `SortRows` | exacto |
| 4 | Asignar clave subrogada | `Sequence` | exacto |
| 5 | Cargar dim_producto | `TableOutput` | exacto |